In [ ]:
%pip install tensorflow

In [ ]:
import os
import time
import json
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


Load dataset

In [ ]:
DATASET_PATH = r"C:\Users\12dem\OneDrive\Desktop\Mtech_assignments\mid 2\DNN\CNN\plantvillage dataset"   # <-- CHANGE THIS

print(os.listdir(DATASET_PATH))


data loading and parting

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.1   # 90/10 split
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

val_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes
print("Number of classes:", NUM_CLASSES)


CNN

In [ ]:
custom_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(),

    layers.GlobalAveragePooling2D(),   # ✅ REQUIRED
    layers.Dense(NUM_CLASSES, activation="softmax")
])

custom_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

custom_model.summary()


Train cnn

In [ ]:
start_time = time.time()

history_custom = custom_model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)

custom_time = time.time() - start_time


In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False   # ✅ REQUIRED

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

transfer_model = models.Model(inputs=base_model.input, outputs=outputs)

transfer_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

transfer_model.summary()


In [ ]:
start_time = time.time()

history_transfer = transfer_model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator
)

transfer_time = time.time() - start_time


In [ ]:
y_true = val_generator.classes

y_pred_custom = np.argmax(custom_model.predict(val_generator), axis=1)
y_pred_transfer = np.argmax(transfer_model.predict(val_generator), axis=1)

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro"),
        "recall": recall_score(y_true, y_pred, average="macro"),
        "f1_score": f1_score(y_true, y_pred, average="macro")
    }

custom_metrics = compute_metrics(y_true, y_pred_custom)
transfer_metrics = compute_metrics(y_true, y_pred_transfer)

print("Custom CNN Metrics:", custom_metrics)
print("Transfer Learning Metrics:", transfer_metrics)


In [ ]:
def get_assignment_results():
    return {
        "custom_cnn": {
            "metrics": custom_metrics,
            "training_time": custom_time
        },
        "transfer_learning": {
            "metrics": transfer_metrics,
            "training_time": transfer_time,
            "base_model": "ResNet50",
            "frozen_layers": len(base_model.layers),
            "trainable_layers": 2
        }
    }

print(json.dumps(get_assignment_results(), indent=4))
